# 10 -- Production Patterns

## What We'll Cover
1. Error handling and retry strategies
2. Timeout configuration for different workloads
3. Rate limiting and backpressure
4. Profile caching for high-throughput apps
5. Logging and monitoring
6. Testing strategies
7. Production deployment checklist


In [ ]:
# 10.1 Comprehensive error handling
from supermemory import (
    Supermemory, APIError, APIConnectionError, APIStatusError,
    AuthenticationError, RateLimitError, PermissionDeniedError,
    NotFoundError, BadRequestError, InternalServerError
)
import time

class RobustClient:
    """Production-ready wrapper with intelligent error handling."""

    def __init__(self, max_retries: int = 3, base_delay: float = 1.0):
        self.client = Supermemory()
        self.max_retries = max_retries
        self.base_delay = base_delay

    def add_with_retry(self, content: str, container_tag: str, **kwargs):
        """Add content with intelligent retry logic."""
        for attempt in range(self.max_retries + 1):
            try:
                return self.client.add(content=content, container_tag=container_tag, **kwargs)
            except RateLimitError:
                if attempt == self.max_retries: raise
                delay = self.base_delay * (2 ** attempt)
                print(f"Rate limited. Retrying in {delay}s...")
                time.sleep(delay)
            except APIConnectionError:
                if attempt == self.max_retries: raise
                delay = self.base_delay * (2 ** attempt)
                print(f"Connection error. Retrying in {delay}s...")
                time.sleep(delay)
            except (BadRequestError, AuthenticationError, PermissionDeniedError) as e:
                print(f"Client error (not retrying): {e}")
                raise
            except InternalServerError:
                if attempt == self.max_retries: raise
                delay = self.base_delay * (2 ** attempt)
                print(f"Server error. Retrying in {delay}s...")
                time.sleep(delay)

print("RobustClient strategy:")
print("  Rate limits: exponential backoff (1s, 2s, 4s)")
print("  Connection errors: retry with backoff")
print("  Client errors (400, 401, 403): no retry, fast fail")
print("  Server errors (500+): retry with backoff")


In [ ]:
# 10.2 Timeout configuration for different workloads
import httpx

# Fast operations: search, profile (sub-300ms normally)
fast_client = Supermemory(timeout=10.0)

# Slow operations: file uploads, URL extraction (1-10 min)
slow_client = Supermemory(
    timeout=httpx.Timeout(600.0, connect=10.0, read=300.0, write=60.0)
)

# Per-request timeouts override global
# fast_client.with_options(timeout=5.0).search.execute(q="quick")

print("Timeout strategies:")
print("  Search/Profile: 10s (normally <1s)")
print("  File upload: 600s (large files need time)")
print("  URL extraction: 300s (page fetch + processing)")
print("  Use with_options() for per-call tuning")


In [ ]:
# 10.3 Rate limiter (token bucket)
from datetime import datetime

class TokenBucket:
    """Token bucket rate limiter for API calls."""
    def __init__(self, rate: float, burst: int):
        self.rate = rate  # tokens per second
        self.burst = burst  # max burst size
        self.tokens = float(burst)
        self.last_update = datetime.now()

    def acquire(self) -> bool:
        """Try to acquire a token. Returns True if successful."""
        now = datetime.now()
        elapsed = (now - self.last_update).total_seconds()
        self.tokens = min(self.burst, self.tokens + elapsed * self.rate)
        self.last_update = now
        if self.tokens >= 1.0:
            self.tokens -= 1.0
            return True
        return False

# Usage: limit to 10 requests per second with burst of 20
limiter = TokenBucket(rate=10.0, burst=20)
print("Rate limiter configured: 10 req/s, burst 20")
print("Use: if limiter.acquire(): make_call() else: backoff()")


In [ ]:
# 10.4 Profile caching for high-throughput applications
from datetime import datetime, timedelta

class CachedProfileService:
    """Cache user profiles to reduce API calls."""

    def __init__(self, ttl_seconds: int = 60):
        self.client = Supermemory()
        self.ttl = ttl_seconds
        self._cache = {}

    def get_profile(self, container_tag: str, query: str = None):
        """Get profile with TTL-based caching."""
        now = datetime.now()
        if container_tag in self._cache:
            cached_data, cached_at = self._cache[container_tag]
            if (now - cached_at).total_seconds() < self.ttl:
                print(f"Cache HIT for {container_tag}")
                return cached_data

        print(f"Cache MISS for {container_tag}")
        profile = self.client.profile(container_tag=container_tag, q=query)
        self._cache[container_tag] = (profile, now)
        return profile

    def invalidate(self, container_tag: str):
        """Force-refresh after adding new content."""
        self._cache.pop(container_tag, None)
        print(f"Cache invalidated for {container_tag}")

# Demo
cache = CachedProfileService(ttl_seconds=30)
p1 = cache.get_profile("user_sarah")
p2 = cache.get_profile("user_sarah")  # Cache hit
print("\nCaching is critical for 100+ concurrent users.")
print("Trade-off: cached profiles may be slightly stale. 30-60s TTL is a good default.")


In [ ]:
# 10.5 Logging and monitoring
import os
import logging

# Enable SDK logging
os.environ["SUPERMEMORY_LOG"] = "info"
# os.environ["SUPERMEMORY_LOG"] = "debug"  # More verbose

# Set up structured logging for your application
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)
logger = logging.getLogger("myapp.supermemory")

class MonitoredClient:
    """Client wrapper with instrumentation."""
    def __init__(self):
        self.client = Supermemory()
        self.stats = {"adds": 0, "searches": 0, "profiles": 0, "errors": 0}

    def add(self, *args, **kwargs):
        try:
            result = self.client.add(*args, **kwargs)
            self.stats["adds"] += 1
            return result
        except Exception as e:
            self.stats["errors"] += 1
            logger.error(f"add() failed: {e}")
            raise

    def profile(self, *args, **kwargs):
        try:
            result = self.client.profile(*args, **kwargs)
            self.stats["profiles"] += 1
            return result
        except Exception as e:
            self.stats["errors"] += 1
            logger.error(f"profile() failed: {e}")
            raise

    def report(self):
        for k, v in self.stats.items():
            print(f"  {k}: {v}")

print("Monitored client ready. Key metrics to track:")
print("  - Call counts (adds, searches, profiles)")
print("  - Error rate and error types")
print("  - Latency percentiles (p50, p95, p99)")
print("  - Rate limit warnings")


In [ ]:
# 10.6 Testing strategies
print("Testing Your Supermemory Integration:")
print()
print("1. UNIT TESTS (mock the SDK)")
print("   from unittest.mock import patch, MagicMock")
print("   @patch('supermemory.Supermemory')")
print("   def test_chatbot_context(mock_sm): ...")
print()
print("2. INTEGRATION TESTS (use self-hosted in CI)")
print("   npx supermemory local &")
print("   Run tests against localhost:6767")
print()
print("3. CONTRACT TESTS")
print("   def test_add_returns_document_id():")
print("       r = client.add(content='test', container_tag='test')")
print("       assert r.id is not None")
print("       assert r.status in ('queued', 'done')")
print()
print("4. PERFORMANCE TESTS")
print("   def test_profile_latency():")
print("       start = time.time()")
print("       client.profile(container_tag='test')")
print("       assert time.time() - start < 1.0")


## 10.7 Production Deployment Checklist

### Security
- [ ] API keys stored in environment variables, never in source code
- [ ] Use `.env` files only for local dev; use secrets manager in production
- [ ] Rotate API keys regularly
- [ ] Use separate API keys for dev/staging/production

### Reliability
- [ ] Implement retry logic with exponential backoff
- [ ] Set appropriate timeouts for each operation type
- [ ] Cache profiles for high-throughput scenarios
- [ ] Graceful degradation: fall back to generic responses if Supermemory is down

### Performance
- [ ] Use `profile()` with `q=` parameter (single call, not profile + search separately)
- [ ] Limit search results to 3-5 items for context injection
- [ ] Use `only_matching_chunks=True` for speed-critical searches
- [ ] Consider AsyncSupermemory for concurrent operations

### Observability
- [ ] Log all API errors with context
- [ ] Track latency percentiles (p50, p95, p99)
- [ ] Monitor rate limit usage
- [ ] Set up alerts for error rate spikes

### Data Management
- [ ] Use `container_tag` consistently across your application
- [ ] Define a metadata schema for your domain
- [ ] Implement data retention policies
- [ ] Test with self-hosted Supermemory before going to production

## 10.8 Key Takeaways

- Handle errors by type: retry server errors, fast-fail on client errors
- Cache profiles for high-throughput apps (30-60s TTL is a good default)
- Use self-hosted Supermemory for CI/testing
- Always have a fallback path when memory is unavailable
- Monitor: you can't improve what you can't measure

---

**Congratulations!** You've completed the Supermemory tutorial.
You now know how to:
- Add and search memories
- Build user profiles that evolve automatically
- Use the knowledge graph for updates, extensions, and inferences
- Filter with metadata and container tags
- Integrate with LLMs (chatbots, agents, RAG)
- Deploy production-ready memory systems

For more: [supermemory.ai/docs](https://supermemory.ai/docs) | [Discord](https://supermemory.link/discord)
